# LeetCode 219 — Contains Duplicate II bằng Python

**Link đề:** https://leetcode.com/problems/contains-duplicate-ii/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sliding Window

```python
class Solution:
    def containsNearbyDuplicate(self, nums: list[int], k: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu điều kiện "trùng **gần nhau**": `nums[i] == nums[j]` và `abs(i - j) <= k`.
2. Đi từ brute force `O(n²)` → quét cửa sổ `O(n·k)` → **hash map lưu chỉ số gần nhất** `O(n)`.
3. Học kỹ thuật **sliding window** (cửa sổ trượt) với `set` có kích thước tối đa `k`.
4. Hiểu vì sao chỉ cần nhớ **lần xuất hiện gần nhất** của mỗi giá trị.
5. Trace, chứng minh bằng invariant, test biên `k = 0`, `k >= n`, và luyện các biến thể.

## 1. Đề bài

Cho mảng `nums` và số nguyên `k`. Trả `True` nếu tồn tại hai chỉ số **khác nhau** `i`, `j` sao cho:
- `nums[i] == nums[j]`, **và**
- `abs(i - j) <= k`.

| `nums` | `k` | Kết quả | Vì sao |
|---|:-:|:-:|---|
| `[1, 2, 3, 1]` | 3 | `True` | `1` ở chỉ số 0 và 3, khoảng cách 3 ≤ 3 |
| `[1, 0, 1, 1]` | 1 | `True` | `1` ở chỉ số 2 và 3, khoảng cách 1 |
| `[1, 2, 3, 1, 2, 3]` | 2 | `False` | mọi cặp trùng đều cách nhau 3 |

**Constraints:** `1 <= len(nums) <= 10^5`; `-10^9 <= nums[i] <= 10^9`; `0 <= k <= 10^5`.

So với bài 217 (Contains Duplicate): thêm ràng buộc **khoảng cách chỉ số**. Chỉ biết "đã thấy" là chưa đủ; cần biết "thấy **ở đâu**".

## 2. Nền tảng Python

### 2.1. `abs()` và khoảng cách chỉ số

`abs(x)` là giá trị tuyệt đối. Khoảng cách giữa hai vị trí là `abs(i - j)`. Khi duyệt trái → phải và `j < i` thì `abs(i - j) = i - j`.

In [1]:
print(abs(-3), abs(3), abs(0))
i, j = 3, 0
print("abs(i - j) =", abs(i - j), "| i - j =", i - j)

3 3 0
abs(i - j) = 3 | i - j = 3


### 2.2. `enumerate` — lấy đồng thời chỉ số và giá trị

In [2]:
for i, x in enumerate([1, 0, 1, 1]):
    print("i =", i, "x =", x)

i = 0 x = 1
i = 1 x = 0
i = 2 x = 1
i = 3 x = 1


### 2.3. `dict` — ghi nhớ "giá trị → chỉ số"

`last = {}`; `last[x] = i` ghi (hoặc **ghi đè**) chỉ số mới nhất của `x`; `x in last` kiểm tra đã từng thấy `x`; `last[x]` đọc chỉ số đã lưu. Tra/ghi trung bình `O(1)`.

In [3]:
last = {}
last[1] = 0
last[0] = 1
print(last)
last[1] = 2            # ghi đè: chỉ giữ vị trí GẦN NHẤT
print(last, "| 1 in last:", 1 in last, "| last[1] =", last[1])

{1: 0, 0: 1}
{1: 2, 0: 1} | 1 in last: True | last[1] = 2


### 2.4. `set` — thêm và **xóa** phần tử

Ngoài `add` và `in`, ở bài này ta cần **xóa**: `s.remove(x)` (lỗi nếu không có `x`) hoặc `s.discard(x)` (không lỗi). Dùng để "trượt" cửa sổ.

In [4]:
window = {1, 2, 3}
window.remove(1)
print(window)
window.discard(99)      # không có 99 -> không lỗi
print(window)
try:
    window.remove(99)
except KeyError as e:
    print("remove phần tử không có -> KeyError:", e)

{2, 3}
{2, 3}
remove phần tử không có -> KeyError: 99


## 3. Cách 1 — Brute Force: thử mọi cặp

Thử mọi cặp `i < j`; nếu bằng nhau và `j - i <= k` → `True`. `O(n²)` — với `n = 10^5` là quá chậm.

In [5]:
def nearby_brute(nums: list[int], k: int) -> bool:
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] == nums[j] and j - i <= k:
                return True
    return False

print(nearby_brute([1, 2, 3, 1], 3), nearby_brute([1, 0, 1, 1], 1), nearby_brute([1, 2, 3, 1, 2, 3], 2))

True True False


## 4. Cách 2 — Chỉ so với `k` phần tử phía trước (cửa sổ quét thô)

Cải tiến nhỏ: với mỗi `i`, chỉ cần so với các chỉ số `j` trong `[i - k, i - 1]` — xa hơn thì không hợp lệ.

`max(0, i - k)` để không có chỉ số âm.

Time `O(n·min(n, k))`. Tốt khi `k` nhỏ, nhưng `k` có thể tới `10^5` → vẫn `O(n²)` trường hợp xấu.

In [6]:
def nearby_window_scan(nums: list[int], k: int) -> bool:
    for i in range(len(nums)):
        for j in range(max(0, i - k), i):
            if nums[j] == nums[i]:
                return True
    return False

print(nearby_window_scan([1, 2, 3, 1], 3), nearby_window_scan([1, 0, 1, 1], 1), nearby_window_scan([1, 2, 3, 1, 2, 3], 2))
print("max(0, 2 - 5) =", max(0, 2 - 5), "| list(range(max(0, 7 - 3), 7)) =", list(range(max(0, 7 - 3), 7)))

True True False
max(0, 2 - 5) = 0 | list(range(max(0, 7 - 3), 7)) = [4, 5, 6]


## 5. Nút thắt & insight — chỉ cần lần xuất hiện **gần nhất**

Tại vị trí `i` với giá trị `x`, giả sử `x` đã xuất hiện ở các chỉ số `j1 < j2 < ... < jm < i`.

Câu hỏi: có `j` nào với `i - j <= k`? Vì `i - j` **nhỏ nhất** khi `j` **lớn nhất**, ta chỉ cần kiểm tra `jm` — lần xuất hiện **gần nhất**. Nếu `jm` quá xa thì những lần cũ hơn càng xa.

⇒ Dùng dict `last[x] = chỉ số gần nhất của x`. Mỗi bước:
1. Nếu `x in last` và `i - last[x] <= k` → `True`.
2. Cập nhật `last[x] = i` (ghi đè vị trí cũ — vị trí cũ không bao giờ còn hữu ích nữa).

In [7]:
nums = [1, 0, 1, 1]
positions = {}
for i, x in enumerate(nums):
    positions.setdefault(x, []).append(i)
print("Mọi vị trí của từng giá trị:", positions)
print("Ở i = 3 (x = 1), các vị trí trước:", positions[1][:-1], "-> gần nhất là", positions[1][-2],
      "khoảng cách", 3 - positions[1][-2])

Mọi vị trí của từng giá trị: {1: [0, 2, 3], 0: [1]}
Ở i = 3 (x = 1), các vị trí trước: [0, 2] -> gần nhất là 2 khoảng cách 1


## 6. Cách 3 — Lời giải chính: hash map "chỉ số gần nhất"

Từng đoạn:

**Đoạn 1:** `last = {}`.

**Đoạn 2:** `for i, x in enumerate(nums):`.

**Đoạn 3 — kiểm tra trước:**
```python
    if x in last and i - last[x] <= k:
        return True
```
`and` ngắn mạch: nếu `x not in last` thì không đọc `last[x]` → không `KeyError`.

**Đoạn 4 — cập nhật sau:**
```python
    last[x] = i
```

**Đoạn 5:** `return False`.

Đây đúng là khuôn "kiểm tra trước, lưu sau" của Two Sum / Contains Duplicate, chỉ đổi thứ được lưu từ "đã thấy" thành "thấy ở đâu".

In [8]:
def nearby_hashmap(nums: list[int], k: int) -> bool:
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False

for nums, k in [([1, 2, 3, 1], 3), ([1, 0, 1, 1], 1), ([1, 2, 3, 1, 2, 3], 2)]:
    print(nums, "k =", k, "->", nearby_hashmap(nums, k))

[1, 2, 3, 1] k = 3 -> True
[1, 0, 1, 1] k = 1 -> True
[1, 2, 3, 1, 2, 3] k = 2 -> False


## 7. TRACE cách 3

### 7.1. Bảng trace `nums = [1, 2, 3, 1, 2, 3]`, `k = 2`

| i | x | `last` trước | `last[x]` | `i - last[x]` | `<= 2`? | Hành động |
|---:|:-:|---|:-:|:-:|:-:|---|
| 0 | 1 | {} | — | — | — | lưu 1:0 |
| 1 | 2 | {1:0} | — | — | — | lưu 2:1 |
| 2 | 3 | {1:0, 2:1} | — | — | — | lưu 3:2 |
| 3 | 1 | {1:0, 2:1, 3:2} | 0 | 3 | không | cập nhật 1:3 |
| 4 | 2 | {1:3, 2:1, 3:2} | 1 | 3 | không | cập nhật 2:4 |
| 5 | 3 | {1:3, 2:4, 3:2} | 2 | 3 | không | cập nhật 3:5 |

Duyệt hết → `False`. Nhánh "trùng nhưng quá xa" xuất hiện 3 lần — đây là nhánh dễ quên khi chỉ học thuộc code.

In [9]:
def trace_nearby(nums: list[int], k: int) -> bool:
    last = {}
    for i, x in enumerate(nums):
        before = dict(last)
        if x in last:
            d = i - last[x]
            if d <= k:
                print(f"i={i}: x={x}, last={before} -> gặp lại ở {last[x]}, khoảng cách {d} <= {k} -> True")
                return True
            print(f"i={i}: x={x}, last={before} -> gặp lại ở {last[x]}, khoảng cách {d} > {k} -> cập nhật {x}:{i}")
        else:
            print(f"i={i}: x={x}, last={before} -> lần đầu thấy, lưu {x}:{i}")
        last[x] = i
    print("Duyệt hết -> False")
    return False

trace_nearby([1, 2, 3, 1, 2, 3], 2)

i=0: x=1, last={} -> lần đầu thấy, lưu 1:0
i=1: x=2, last={1: 0} -> lần đầu thấy, lưu 2:1
i=2: x=3, last={1: 0, 2: 1} -> lần đầu thấy, lưu 3:2
i=3: x=1, last={1: 0, 2: 1, 3: 2} -> gặp lại ở 0, khoảng cách 3 > 2 -> cập nhật 1:3
i=4: x=2, last={1: 3, 2: 1, 3: 2} -> gặp lại ở 1, khoảng cách 3 > 2 -> cập nhật 2:4
i=5: x=3, last={1: 3, 2: 4, 3: 2} -> gặp lại ở 2, khoảng cách 3 > 2 -> cập nhật 3:5
Duyệt hết -> False


False

### 7.2. Trace ví dụ 2 — `[1, 0, 1, 1]`, `k = 1`

Ở `i = 2`, `1` cách lần trước 2 > 1 nên **không** trả `True`, nhưng ta **cập nhật** `last[1] = 2`. Nhờ cập nhật đó, ở `i = 3` khoảng cách chỉ còn 1 → `True`. Nếu quên cập nhật khi khoảng cách quá xa, ta sẽ trả sai `False`.

In [10]:
trace_nearby([1, 0, 1, 1], 1)

i=0: x=1, last={} -> lần đầu thấy, lưu 1:0
i=1: x=0, last={1: 0} -> lần đầu thấy, lưu 0:1
i=2: x=1, last={1: 0, 0: 1} -> gặp lại ở 0, khoảng cách 2 > 1 -> cập nhật 1:2
i=3: x=1, last={1: 2, 0: 1} -> gặp lại ở 2, khoảng cách 1 <= 1 -> True


True

## 8. Cách 4 — Sliding window bằng `set` kích thước ≤ k

Góc nhìn khác: điều kiện `i - j <= k` nghĩa là `nums[j]` nằm trong **cửa sổ** `k` phần tử ngay trước `i`: chỉ số `[i - k, i - 1]`.

Giữ một `set` `window` chứa **đúng** các giá trị trong cửa sổ đó:
1. Nếu `x in window` → `True`.
2. Thêm `x` vào `window`.
3. Nếu `len(window) > k` → xóa phần tử **rơi khỏi** cửa sổ: `nums[i - k]`.

```text
k = 2, nums = [1, 2, 3, 1]
i=0: window {}      -> thêm 1 -> {1}
i=1: window {1}     -> thêm 2 -> {1,2}
i=2: window {1,2}   -> thêm 3 -> {1,2,3} quá k -> bỏ nums[0]=1 -> {2,3}
i=3: x=1 không trong {2,3} -> ...
```

Vì sao set không bị "mất" phần tử trùng? Nếu trong cửa sổ có hai giá trị bằng nhau thì ta đã trả `True` trước đó — nên mọi giá trị trong `window` luôn **khác nhau**, và `len(window)` đúng bằng số phần tử của cửa sổ.

In [11]:
def nearby_sliding_window(nums: list[int], k: int) -> bool:
    window = set()
    for i, x in enumerate(nums):
        if x in window:
            return True
        window.add(x)
        if len(window) > k:
            window.remove(nums[i - k])
    return False

for nums, k in [([1, 2, 3, 1], 3), ([1, 0, 1, 1], 1), ([1, 2, 3, 1, 2, 3], 2), ([1, 1], 0)]:
    print(nums, "k =", k, "->", nearby_sliding_window(nums, k))

[1, 2, 3, 1] k = 3 -> True
[1, 0, 1, 1] k = 1 -> True
[1, 2, 3, 1, 2, 3] k = 2 -> False
[1, 1] k = 0 -> False


In [12]:
def trace_sliding(nums: list[int], k: int) -> bool:
    window = set()
    for i, x in enumerate(nums):
        if x in window:
            print(f"i={i}: x={x} có trong cửa sổ {sorted(window)} -> True")
            return True
        window.add(x)
        msg = f"i={i}: thêm {x} -> {sorted(window)}"
        if len(window) > k:
            out = nums[i - k]
            window.remove(out)
            msg += f" | quá {k} phần tử, bỏ nums[{i - k}]={out} -> {sorted(window)}"
        print(msg)
    print("Duyệt hết -> False")
    return False

trace_sliding([1, 2, 3, 1, 2, 3], 2)

i=0: thêm 1 -> [1]
i=1: thêm 2 -> [1, 2]
i=2: thêm 3 -> [1, 2, 3] | quá 2 phần tử, bỏ nums[0]=1 -> [2, 3]
i=3: thêm 1 -> [1, 2, 3] | quá 2 phần tử, bỏ nums[1]=2 -> [1, 3]
i=4: thêm 2 -> [1, 2, 3] | quá 2 phần tử, bỏ nums[2]=3 -> [1, 2]
i=5: thêm 3 -> [1, 2, 3] | quá 2 phần tử, bỏ nums[3]=1 -> [2, 3]
Duyệt hết -> False


False

### 8.1. Vì sao xóa đúng `nums[i - k]`?

Sau khi thêm `nums[i]`, cửa sổ muốn giữ cho vòng **sau** (`i + 1`) là các chỉ số `[i + 1 - k, i]` — tức `k` phần tử. Phần tử vừa rơi ra là chỉ số `i - k`. Khi `i < k` thì `len(window) <= k` nên không xóa gì (cửa sổ chưa đầy).

Trường hợp `k = 0`: thêm xong `len = 1 > 0` → xóa ngay → cửa sổ luôn rỗng → luôn `False`. Đúng, vì `abs(i - j) <= 0` buộc `i == j`, mà đề cần hai chỉ số **khác nhau**.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def containsNearbyDuplicate(self, nums: list[int], k: int) -> bool:
        last = {}
        for i, x in enumerate(nums):
            if x in last and i - last[x] <= k:
                return True
            last[x] = i
        return False

sol = Solution()
print(sol.containsNearbyDuplicate([1, 2, 3, 1], 3), sol.containsNearbyDuplicate([1, 0, 1, 1], 1),
      sol.containsNearbyDuplicate([1, 2, 3, 1, 2, 3], 2))

True True False


Bản sliding window (bộ nhớ `O(min(n, k))`):

```python
class Solution:
    def containsNearbyDuplicate(self, nums: list[int], k: int) -> bool:
        window = set()
        for i, x in enumerate(nums):
            if x in window:
                return True
            window.add(x)
            if len(window) > k:
                window.remove(nums[i - k])
        return False
```

## 10. Vì sao đúng?

**Cách 3 (hash map)** — invariant ở đầu vòng `i`:

> Với mỗi giá trị `v` đã gặp trong `nums[0..i-1]`, `last[v]` là chỉ số **lớn nhất** `j < i` có `nums[j] == v`; và chưa có cặp hợp lệ nào trong `nums[0..i-1]`.

- **Ban đầu:** rỗng → đúng.
- **Bước `i`:** cặp hợp lệ mới (nếu có) phải dùng `i` làm chỉ số lớn hơn. Cặp tốt nhất với `i` là `(last[x], i)` vì `last[x]` gần `i` nhất. Nếu khoảng cách đó `<= k` → trả `True` đúng; nếu `> k` thì mọi `j` khác của `x` còn xa hơn → không có cặp với `i`. Gán `last[x] = i` giữ invariant.
- **Kết thúc:** không có cặp hợp lệ nào → `False` đúng.

**Cách 4 (window)** — invariant: đầu vòng `i`, `window` = tập giá trị `nums[max(0, i-k) .. i-1]` (đôi một khác nhau). `x in window` ⇔ tồn tại `j` trong khoảng đó với `nums[j] == x` ⇔ có cặp hợp lệ kết thúc tại `i`.

## 11. Time / Space — so sánh

Gọi `n = len(nums)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` | TLE với `n = 10^5` |
| 2. Quét `k` phần tử trước | `O(n·min(n, k))` | `O(1)` | tốt khi `k` rất nhỏ |
| 3. Hash map chỉ số gần nhất | `O(n)` trung bình | `O(n)` (số giá trị khác nhau) | **chuẩn**, dễ giải thích |
| 4. Sliding window set | `O(n)` trung bình | `O(min(n, k))` | tiết kiệm bộ nhớ khi `k` nhỏ |

In [14]:
import time, random
rng = random.Random(219)
nums = rng.sample(range(-10**9, 10**9), 10**5)      # không trùng -> worst case
k = 10**5
for name, f in [("hashmap", nearby_hashmap), ("window", nearby_sliding_window)]:
    t0 = time.perf_counter(); r = f(nums, k); t1 = time.perf_counter()
    print(f"{name:<8} n=10^5, k=10^5 -> {r} trong {t1 - t0:.4f}s")
t0 = time.perf_counter(); nearby_window_scan(nums[:2000], k); t1 = time.perf_counter()
print(f"quét thô chỉ với 2000 phần tử: {t1 - t0:.4f}s")

hashmap  n=10^5, k=10^5 -> False trong 0.0148s
window   n=10^5, k=10^5 -> False trong 0.0133s


quét thô chỉ với 2000 phần tử: 0.0615s


## 12. Bộ test

Nhóm test: 3 ví dụ đề; `k = 0` (luôn `False`); `k >= n`; trùng kề nhau; trùng đúng khoảng cách `k` và `k + 1`; một phần tử; nhánh "lần đầu quá xa, lần sau đủ gần".

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
f = sol.containsNearbyDuplicate
check("ví dụ 1", f([1, 2, 3, 1], 3), True)
check("ví dụ 2", f([1, 0, 1, 1], 1), True)
check("ví dụ 3", f([1, 2, 3, 1, 2, 3], 2), False)
check("k = 0", f([1, 1], 0), False)
check("1 phần tử", f([5], 3), False)
check("trùng kề nhau, k = 1", f([7, 7], 1), True)
check("khoảng cách đúng bằng k", f([1, 2, 3, 4, 1], 4), True)
check("khoảng cách k + 1", f([1, 2, 3, 4, 1], 3), False)
check("k lớn hơn n", f([1, 2, 1], 10**5), True)
check("xa rồi gần (cần cập nhật)", f([9, 0, 0, 0, 9, 1, 9], 2), True)
check("không trùng, n lớn", f(list(range(10**5)), 10**5), False)

PASS ví dụ 1: True
PASS ví dụ 2: True
PASS ví dụ 3: False
PASS k = 0: False
PASS 1 phần tử: False
PASS trùng kề nhau, k = 1: True
PASS khoảng cách đúng bằng k: True
PASS khoảng cách k + 1: False
PASS k lớn hơn n: True
PASS xa rồi gần (cần cập nhật): True
PASS không trùng, n lớn: False


In [16]:
rng = random.Random(2019)
fns = [nearby_brute, nearby_window_scan, nearby_hashmap, nearby_sliding_window, Solution().containsNearbyDuplicate]
for _ in range(4000):
    nums = [rng.randint(0, 5) for _ in range(rng.randint(1, 12))]
    k = rng.randint(0, 6)
    res = {g(nums, k) for g in fns}
    assert len(res) == 1, (nums, k, res)
print("5 cách khớp nhau trên 4000 input ngẫu nhiên")

5 cách khớp nhau trên 4000 input ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ cập nhật `last[x]` khi chưa có (giữ vị trí **cũ nhất**)

```python
if x not in last:
    last[x] = i       # SAI: không ghi đè -> luôn so với lần đầu tiên
```

In [17]:
def nearby_bug_keep_first(nums, k):
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        if x not in last:
            last[x] = i
    return False

print("Bản lỗi [1,0,1,1], k=1:", nearby_bug_keep_first([1, 0, 1, 1], 1), "(đúng phải là True)")

Bản lỗi [1,0,1,1], k=1: False (đúng phải là True)


### Lỗi 2 — Dùng `<` thay vì `<=`

Đề nói `abs(i - j) <= k`. Với `[1, 2, 3, 1]`, `k = 3`: khoảng cách 3 phải được tính là hợp lệ.

In [18]:
def nearby_bug_strict(nums, k):
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] < k:
            return True
        last[x] = i
    return False

print("Bản lỗi '<' với [1,2,3,1], k=3:", nearby_bug_strict([1, 2, 3, 1], 3), "(đúng phải là True)")

Bản lỗi '<' với [1,2,3,1], k=3: False (đúng phải là True)


### Lỗi 3 — Sliding window xóa sai phần tử

Xóa `nums[i - k - 1]` hoặc xóa **trước** khi kiểm tra → cửa sổ lệch một phần tử: hoặc bỏ sót cặp cách đúng `k`, hoặc nhận nhầm cặp cách `k + 1`.

### Lỗi 4 — Đọc `last[x]` khi `x` chưa có

`if i - last[x] <= k` mà không kiểm tra `x in last` trước → `KeyError`. Dùng `and` ngắn mạch hoặc `last.get(x)`.

### Lỗi 5 — Quên rằng `i` và `j` phải **khác nhau**

Với `k = 0`, nếu viết vòng lặp brute force với `j` chạy từ `i` thì `nums[i] == nums[i]` → luôn `True`. Phải `j` từ `i + 1`.

In [19]:
def nearby_bug_window_off_by_one(nums, k):
    window = set()
    for i, x in enumerate(nums):
        if x in window:
            return True
        window.add(x)
        if len(window) > k + 1:                 # SAI: giữ k + 1 phần tử
            window.remove(nums[i - k - 1])
    return False

print("Bản lỗi cửa sổ, [1,2,1], k=1:", nearby_bug_window_off_by_one([1, 2, 1], 1), "(đúng phải là False)")

Bản lỗi cửa sổ, [1,2,1], k=1: True (đúng phải là False)


## 14. Mẹo nhận dạng pattern

- "Hai phần tử bằng nhau **trong khoảng cách ≤ k**", "trong mọi đoạn dài k" → **sliding window** hoặc **dict chỉ số gần nhất**.
- Cần biết "ở đâu" chứ không chỉ "có chưa" → **dict `giá_trị → chỉ số`** thay vì set.
- "Gần nhất" là đủ khi điều kiện đơn điệu theo khoảng cách (càng gần càng tốt).
- Họ bài:
  - 217 Contains Duplicate (không giới hạn khoảng cách).
  - 220 Contains Duplicate III: `abs(nums[i] - nums[j]) <= t` và `abs(i - j) <= k` → sliding window + **bucket**.
  - 3 Longest Substring Without Repeating Characters: dict chỉ số gần nhất + cửa sổ.
  - 1 Two Sum: dict `giá_trị → chỉ số`.

## 15. Biến thể

### 15.1. Khoảng cách **nhỏ nhất** giữa hai phần tử bằng nhau
Thay vì dừng sớm, lấy `min` của `i - last[x]` qua mọi lần gặp lại.

### 15.2. Đếm số cặp `(i, j)` hợp lệ
Cần nhớ **mọi** vị trí trong cửa sổ, không chỉ gần nhất → dùng `Counter` trên cửa sổ trượt.

### 15.3. Contains Duplicate III (ý tưởng bucket)
Chia trục số thành các "xô" độ rộng `t + 1`; hai số chênh ≤ `t` nằm cùng xô hoặc hai xô kề nhau.

In [20]:
def min_equal_distance(nums):
    last, best = {}, None
    for i, x in enumerate(nums):
        if x in last:
            d = i - last[x]
            best = d if best is None else min(best, d)
        last[x] = i
    return best

from collections import Counter

def count_nearby_pairs(nums, k):
    window = Counter()
    total = 0
    for i, x in enumerate(nums):
        total += window[x]                 # số phần tử bằng x trong k vị trí trước
        window[x] += 1
        if i - k >= 0:
            window[nums[i - k]] -= 1       # phần tử rơi khỏi cửa sổ cho vòng sau
    return total

print("Khoảng cách nhỏ nhất [1,2,3,1,2,1]:", min_equal_distance([1, 2, 3, 1, 2, 1]))
print("Số cặp hợp lệ [1,1,1], k=1:", count_nearby_pairs([1, 1, 1], 1), "| k=2:", count_nearby_pairs([1, 1, 1], 2))

Khoảng cách nhỏ nhất [1,2,3,1,2,1]: 2
Số cặp hợp lệ [1,1,1], k=1: 2 | k=2: 3


## 16. Bài tập tự luyện

### Bài 1 — Tính tay
Trace cách hash map cho `nums = [4, 1, 2, 4, 1]`, `k = 2`. Kết quả là gì? Dừng ở đâu (nếu dừng)?

### Bài 2 — Dự đoán
Với `k = 0`, cách hash map có bao giờ trả `True` không? Vì sao?

### Bài 3 — TODO: viết cách hash map

In [21]:
def exercise_nearby(nums: list[int], k: int) -> bool:
    # TODO 1: last = {}
    # TODO 2: kiểm tra x in last và i - last[x] <= k
    # TODO 3: luôn cập nhật last[x] = i
    pass

# assert exercise_nearby([1, 2, 3, 1], 3) is True
# assert exercise_nearby([1, 2, 3, 1, 2, 3], 2) is False

### Bài 4 — TODO: viết cách sliding window

In [22]:
def exercise_nearby_window(nums: list[int], k: int) -> bool:
    # TODO: set window; kiểm tra, thêm, nếu len > k thì bỏ nums[i - k]
    pass

# assert exercise_nearby_window([1, 0, 1, 1], 1) is True
# assert exercise_nearby_window([1, 1], 0) is False

### Bài 5 — Biến thể: trả về cặp chỉ số `(j, i)` đầu tiên thỏa mãn (theo `i` nhỏ nhất), không có thì `None`
`first_nearby_pair([1, 0, 1, 1], 1)` → `(2, 3)`.

In [23]:
def exercise_first_nearby_pair(nums: list[int], k: int):
    # TODO
    pass

# assert exercise_first_nearby_pair([1, 0, 1, 1], 1) == (2, 3)
# assert exercise_first_nearby_pair([1, 2, 3], 5) is None

### Bài 6 — Mini challenge: chuỗi có ký tự lặp lại trong vòng `k` ký tự không?
`has_nearby_repeat("abcab", 2)` → `False` (các ký tự lặp cách nhau 3); `has_nearby_repeat("abca", 3)` → `True`. Áp dụng lại khuôn cho chuỗi.

In [24]:
def exercise_has_nearby_repeat(s: str, k: int) -> bool:
    # TODO
    pass

# assert exercise_has_nearby_repeat("abcab", 2) is False
# assert exercise_has_nearby_repeat("abca", 3) is True

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Ghi `last` mỗi bước | khoảng cách 4 ở i = 3 là `3 - 0` | ở i = 4, `1` cách `4 - 1 = 3` |
| 2 | `i - last[x]` nhỏ nhất bằng mấy? | chỉ số khác nhau → ≥ 1 | `1 <= 0` sai |
| 3 | Kiểm tra trước, lưu sau | `x in last and ...` | `last[x] = i` ngoài `if` |
| 4 | Cửa sổ k phần tử trước | `window.add(x)` | `if len(window) > k: window.remove(nums[i - k])` |
| 5 | Như bài 3, trả tuple | `return (last[x], i)` | |
| 6 | Chuỗi duyệt như list | `enumerate(s)` | dùng lại hàm bài 3 |

## Đáp án tham khảo

**Bài 1:** `i=0: lưu 4:0` → `i=1: lưu 1:1` → `i=2: lưu 2:2` → `i=3: 4 cách 3 > 2, cập nhật 4:3` → `i=4: 1 cách 3 > 2, cập nhật 1:4` → `False`.

**Bài 2:** Không bao giờ. Hai chỉ số khác nhau nên `i - last[x] >= 1 > 0`.

In [25]:
def answer_nearby(nums, k):
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False

def answer_nearby_window(nums, k):
    window = set()
    for i, x in enumerate(nums):
        if x in window:
            return True
        window.add(x)
        if len(window) > k:
            window.remove(nums[i - k])
    return False

def answer_first_nearby_pair(nums, k):
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return (last[x], i)
        last[x] = i
    return None

def answer_has_nearby_repeat(s, k):
    return answer_nearby(list(s), k)

assert trace_nearby([4, 1, 2, 4, 1], 2) is False
assert answer_nearby([1, 1, 2, 2, 1], 0) is False          # bài 2: k = 0 không bao giờ True
assert answer_nearby([1, 2, 3, 1], 3) is True and answer_nearby([1, 2, 3, 1, 2, 3], 2) is False
assert answer_nearby_window([1, 0, 1, 1], 1) is True and answer_nearby_window([1, 1], 0) is False
assert answer_first_nearby_pair([1, 0, 1, 1], 1) == (2, 3) and answer_first_nearby_pair([1, 2, 3], 5) is None
assert answer_has_nearby_repeat("abcab", 2) is False and answer_has_nearby_repeat("abca", 3) is True
print("Đáp án tham khảo PASS toàn bộ!")

i=0: x=4, last={} -> lần đầu thấy, lưu 4:0
i=1: x=1, last={4: 0} -> lần đầu thấy, lưu 1:1
i=2: x=2, last={4: 0, 1: 1} -> lần đầu thấy, lưu 2:2
i=3: x=4, last={4: 0, 1: 1, 2: 2} -> gặp lại ở 0, khoảng cách 3 > 2 -> cập nhật 4:3
i=4: x=1, last={4: 3, 1: 1, 2: 2} -> gặp lại ở 1, khoảng cách 3 > 2 -> cập nhật 1:4
Duyệt hết -> False
Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
last = {}                                   # giá_trị -> chỉ số GẦN NHẤT
for i, x in enumerate(nums):
    if x in last and i - last[x] <= k:      # kiểm tra trước
        return True
    last[x] = i                             # luôn cập nhật (ghi đè)
return False
```

```python
window = set()                              # sliding window k phần tử trước
for i, x in enumerate(nums):
    if x in window: return True
    window.add(x)
    if len(window) > k: window.remove(nums[i - k])
return False
```

**Checklist:**
- [ ] Dùng `<= k`, không phải `< k`.
- [ ] Luôn cập nhật `last[x] = i`, kể cả khi khoảng cách quá xa.
- [ ] Sliding window: xóa đúng `nums[i - k]` sau khi thêm.
- [ ] Đã thử `k = 0`, `k >= n`, khoảng cách đúng bằng `k`.
- [ ] Tên method `containsNearbyDuplicate`, trả `bool`.